Three models are evaluated:

1. Text model
2. Image model
3. Structured model

The output is prediction coverage and average prediction-set size.

In [1]:
!pip -q install -U mapie
import os
import glob
import joblib
import numpy as np
import pandas as pd

from mapie.classification import SplitConformalClassifier
from mapie.metrics.classification import (
    classification_coverage_score,
    classification_mean_width_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

from mapie.classification import (
    SplitConformalClassifier,
    CrossConformalClassifier
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.6/555.6 kB 3.4 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount(
    '/content/drive'
)

base_path = (
    '/content/drive/MyDrive/'
    'dissertation_project/data'
)

processed_path = (
    f'{base_path}/processed'
)

model_path = (
    f'{base_path}/models'
)

print(
    "Processed:",
    processed_path
)

print(
    "Models:",
    model_path
)

Mounted at /content/drive
Processed: /content/drive/MyDrive/dissertation_project/data/processed
Models: /content/drive/MyDrive/dissertation_project/data/models


In [3]:
model_files = glob.glob(
    f'{model_path}/*'
)

print(
    "Available model files:"
)

for file in model_files:
    print(
        os.path.basename(file)
    )

Available model files:
bioclinicalbert_text_encoder.pt
vit_image_encoder.pt
fusion_scaler.pkl
multimodal_fusion_layer1.pt
structured_scaler.pkl
structured_mlp.pt


In [4]:
csv_files = glob.glob(
    f'{processed_path}/*.csv'
)

print(
    "Processed CSV files:"
)

for file in csv_files:
    print(
        os.path.basename(file)
    )

Processed CSV files:
matched_final.csv
lab_feature_dictionary.csv
2_image_reference.csv
1_structured_reference.csv
3_text_reference.csv
structured_processed.csv
text_processed.csv
test_calibration_ids.csv
test_calibration_pool.csv
text_baseline_metrics.csv
text_training_history.csv
text_features_heldout.csv
text_features_development.csv
image_baseline_metrics.csv
image_training_history.csv
image_features_heldout.csv
image_features_development.csv
fusion_layer1_metrics.csv
fusion_layer1_training_history.csv
fusion_features_layer1.csv
fusion_features_layer1_heldout.csv
mapie_conformal_results.csv
structured_baseline_metrics.csv
structured_training_history.csv
structured_features_mlp.csv
structured_features_mlp_heldout.csv


In [5]:
calibration_candidates = []

for file in csv_files:

    try:

        df = pd.read_csv(
            file,
            nrows=5
        )

        if len(df.columns) > 0:

            calibration_candidates.append(
                file
            )

    except:

        pass

print(
    "Candidate files found:",
    len(calibration_candidates)
)

Candidate files found: 26


In [6]:
calibration_file = f'{processed_path}/test_calibration_pool.csv'

print("Selected calibration file:")
print(calibration_file)

Selected calibration file:
/content/drive/MyDrive/dissertation_project/data/processed/test_calibration_pool.csv


In [7]:
if calibration_file is None:
    raise FileNotFoundError(
        "Calibration/687-record CSV was not found. "
        "Check the processed folder."
    )

calibration_df = pd.read_csv(calibration_file)

print("Calibration records:", len(calibration_df))

display(calibration_df.head())

Calibration records: 687


,subject_id,study_id,hadm_id,anchor_age,length_of_stay_hours,glucose,creatinine,sodium,potassium,hemoglobin,...,race_UNABLE TO OBTAIN,race_UNKNOWN,race_WHITE,race_WHITE - BRAZILIAN,race_WHITE - EASTERN EUROPEAN,race_WHITE - OTHER EUROPEAN,race_WHITE - RUSSIAN,report_text,image_available,text_available
0,10541652,53372346,21690356,1.040070,-0.591910,5.645434,-0.532377,-0.925329,0.930455,-0.211042,...,-0.047727,-0.240192,-1.287247,-0.052295,-0.036953,-0.149348,-0.130789,FINDINGS: The heart size is mildly enlarged. T...,True,True
1,17397284,52626730,26043254,-0.786119,-0.312028,-0.743815,-0.750588,0.156838,0.245203,-0.191119,...,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789,FINDINGS: IMPRESSION: In comparison with the s...,True,True
2,16074663,55540339,26711273,-0.846992,-0.521324,-1.126125,-0.076413,-0.834199,-1.677924,-1.579093,...,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789,FINDINGS: Cardiac silhouette size is normal. M...,True,True
3,16550115,54653619,24557244,0.066103,-0.361023,0.146931,-0.662652,-0.131198,-2.113706,-1.541144,...,-0.047727,-0.240192,0.776852,-0.052295,-0.036953,-0.149348,-0.130789,FINDINGS: IMPRESSION: Heart size and mediastin...,True,True
4,18879361,59119252,27354255,1.283562,-0.140027,0.298308,-0.288282,0.724121,-0.669940,-0.888426,...,-0.047727,4.163332,-1.287247,-0.052295,-0.036953,-0.149348,-0.130789,FINDINGS: IMPRESSION: Comparison ___. The moni...,True,True


In [8]:
# Identify disease label columns

metadata_columns = [
    'subject_id',
    'study_id',
    'hadm_id',
    'anchor_age',
    'length_of_stay_hours'
]

# Known CheXpert/MIMIC-CXR disease labels
disease_labels = [
    'Atelectasis',
    'Cardiomegaly',
    'Consolidation',
    'Edema',
    'Enlarged Cardiomediastinum',
    'Fracture',
    'Lung Lesion',
    'Lung Opacity',
    'No Finding',
    'Pleural Effusion',
    'Pleural Other',
    'Pneumonia',
    'Pneumothorax',
    'Support Devices'
]

available_labels = [
    col
    for col in disease_labels
    if col in calibration_df.columns
]

print("Disease label columns found:")
print(available_labels)

print(
    "\nNumber of available labels:",
    len(available_labels)
)

Disease label columns found:
['Atelectasis', 'Cardiomegaly', 'Edema', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Support Devices']

Number of available labels: 7


In [9]:
# Inspect disease labels

if len(available_labels) == 0:

    raise ValueError(
        "No disease label columns were found "
        "in the calibration dataset."
    )

print("Label distribution:")

display(
    calibration_df[
        available_labels
    ].sum()
    .sort_values(
        ascending=False
    )
)

Label distribution:


,0
Support Devices,266.0
Lung Opacity,178.0
No Finding,165.0
Pleural Effusion,160.0
Cardiomegaly,139.0
Atelectasis,122.0
Edema,26.0


In [10]:
print(list(df.columns))

['study_id', 'structured_feature_0', 'structured_feature_1', 'structured_feature_2', 'structured_feature_3', 'structured_feature_4', 'structured_feature_5', 'structured_feature_6', 'structured_feature_7', 'structured_feature_8', 'structured_feature_9', 'structured_feature_10', 'structured_feature_11', 'structured_feature_12', 'structured_feature_13', 'structured_feature_14', 'structured_feature_15', 'structured_feature_16', 'structured_feature_17', 'structured_feature_18', 'structured_feature_19', 'structured_feature_20', 'structured_feature_21', 'structured_feature_22', 'structured_feature_23', 'structured_feature_24', 'structured_feature_25', 'structured_feature_26', 'structured_feature_27', 'structured_feature_28', 'structured_feature_29', 'structured_feature_30', 'structured_feature_31', 'structured_feature_32', 'structured_feature_33', 'structured_feature_34', 'structured_feature_35', 'structured_feature_36', 'structured_feature_37', 'structured_feature_38', 'structured_feature_39

In [11]:
# Select one disease for the prototype MAPIE demonstration

TARGET_LABEL = 'Pleural Effusion'

if TARGET_LABEL not in available_labels:

    raise ValueError(
        f"{TARGET_LABEL} is not available."
    )

y_calibration = (
    calibration_df[TARGET_LABEL]
    .fillna(0)
    .astype(int)
    .values
)

print(
    "Selected target:",
    TARGET_LABEL
)

print(
    "Number of calibration records:",
    len(y_calibration)
)

print(
    "Class distribution:"
)

print(
    pd.Series(
        y_calibration
    ).value_counts()
)

Selected target: Pleural Effusion
Number of calibration records: 687
Class distribution:
 0    455
 1    196
-1     36
Name: count, dtype: int64


In [12]:
# Load Layer 1 fusion features

fusion_feature_file = (
    f'{processed_path}/fusion_features_layer1.csv'
)

fusion_features = pd.read_csv(
    fusion_feature_file
)

print(
    "Fusion features shape:",
    fusion_features.shape
)

display(
    fusion_features.head()
)

Fusion features shape: (1513, 129)


,study_id,fusion_feature_0,fusion_feature_1,fusion_feature_2,fusion_feature_3,fusion_feature_4,fusion_feature_5,fusion_feature_6,fusion_feature_7,fusion_feature_8,...,fusion_feature_118,fusion_feature_119,fusion_feature_120,fusion_feature_121,fusion_feature_122,fusion_feature_123,fusion_feature_124,fusion_feature_125,fusion_feature_126,fusion_feature_127
0,58630288,0.000000,0.0,0.0,20.240543,0.0,0.0,0.0,0.0,209.73631,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,35.595055,0.0,240.43594
1,58239923,0.000000,0.0,0.0,44.328487,0.0,0.0,0.0,0.0,248.49486,...,0.000000,0.000000,0.0,0.0,0.000000,0.945898,0.0,13.081628,0.0,276.55440
2,53321493,0.000000,0.0,0.0,23.440650,0.0,0.0,0.0,0.0,225.65085,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,43.213104,0.0,214.85527
3,56501836,16.122736,0.0,0.0,4.782885,0.0,0.0,0.0,0.0,0.00000,...,1.075445,46.294983,0.0,0.0,38.785873,0.000000,0.0,23.156618,0.0,0.00000
4,50650870,0.000000,0.0,0.0,40.734177,0.0,0.0,0.0,0.0,229.50403,...,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,17.085352,0.0,243.49706


In [13]:
print(
    "Fusion feature columns:"
)

print(
    fusion_features.columns.tolist()[:20]
)

print(
    "\nTotal columns:",
    len(fusion_features.columns)
)

Fusion feature columns:
['study_id', 'fusion_feature_0', 'fusion_feature_1', 'fusion_feature_2', 'fusion_feature_3', 'fusion_feature_4', 'fusion_feature_5', 'fusion_feature_6', 'fusion_feature_7', 'fusion_feature_8', 'fusion_feature_9', 'fusion_feature_10', 'fusion_feature_11', 'fusion_feature_12', 'fusion_feature_13', 'fusion_feature_14', 'fusion_feature_15', 'fusion_feature_16', 'fusion_feature_17', 'fusion_feature_18']

Total columns: 129


In [14]:
# Keep only numeric fusion features

X = fusion_features.select_dtypes(
    include=[np.number]
).copy()

print(
    "Numeric feature shape:",
    X.shape
)

Numeric feature shape: (1513, 129)


In [15]:
n = min(
    len(calibration_df),
    len(X)
)

X = X.iloc[:n].values

y = (
    calibration_df[
        TARGET_LABEL
    ]
    .fillna(0)
    .replace(-1, 0)
    .astype(int)
    .iloc[:n]
    .values
)

print(
    "Records used:",
    n
)

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "Classes:",
    np.unique(y)
)

Records used: 687
X shape: (687, 129)
y shape: (687,)
Classes: [0 1]


In [16]:
# training a simple regression model
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=500,
    random_state=42
)

classifier.fit(
    X,
    y
)

print(
    "Classifier fitted successfully."
)

Classifier fitted successfully.


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [17]:
# MAPIE
from mapie.classification import (
    SplitConformalClassifier
)

confidence_level = 0.90

mapie_model = SplitConformalClassifier(
    estimator=classifier,
    confidence_level=confidence_level,
    conformity_score="lac",
    prefit=True
)

mapie_model.conformalize(
    X,
    y
)

print(
    "MAPIE calibration completed."
)

MAPIE calibration completed.


In [18]:
# Generate prediction sets
y_pred, prediction_sets = (
    mapie_model.predict_set(
        X
    )
)

print(
    "Predictions generated:",
    len(y_pred)
)

print(
    "Prediction-set shape:",
    prediction_sets.shape
)

Predictions generated: 687
Prediction-set shape: (687, 2, 1)


In [19]:
# Calculate coverage
from mapie.metrics.classification import (
    classification_coverage_score,
    classification_mean_width_score
)

coverage = (
    classification_coverage_score(
        y,
        prediction_sets
    )
)

set_size = (
    classification_mean_width_score(
        prediction_sets
    )
)

print(
    "Empirical coverage:",
    round(
        float(np.asarray(coverage).ravel()[0]),
        4
    )
)

print(
    "Average prediction-set size:",
    round(
        float(np.asarray(set_size).ravel()[0]),
        4
    )
)

Empirical coverage: 0.9025
Average prediction-set size: 1.4687


In [20]:
# Results table
mapie_results = pd.DataFrame({

    'Method': [
        'Split Conformal'
    ],

    'Target': [
        TARGET_LABEL
    ],

    'Confidence Level': [
        confidence_level
    ],

    'Empirical Coverage': [
        float(
            np.asarray(
                coverage
            ).ravel()[0]
        )
    ],

    'Average Prediction Set Size': [
        float(
            np.asarray(
                set_size
            ).ravel()[0]
        )
    ]
})

display(
    mapie_results
)

,Method,Target,Confidence Level,Empirical Coverage,Average Prediction Set Size
0,Split Conformal,Pleural Effusion,0.9,0.902475,1.468705


In [21]:
# Show example prediction sets
print(
    "Example prediction sets:"
)

for i in range(
    min(5, len(y_pred))
):

    print(
        "Sample",
        i + 1,
        "| True:",
        y[i],
        "| Prediction:",
        y_pred[i],
        "| Set:",
        np.where(
            prediction_sets[
                i, :, 0
            ]
        )[0]
    )

Example prediction sets:
Sample 1 | True: 1 | Prediction: 0 | Set: [0 1]
Sample 2 | True: 1 | Prediction: 0 | Set: [0]
Sample 3 | True: 0 | Prediction: 0 | Set: [0]
Sample 4 | True: 0 | Prediction: 0 | Set: [0]
Sample 5 | True: 0 | Prediction: 0 | Set: [0]


In [22]:
# Save result
output_file = (
    f'{processed_path}/'
    'mapie_conformal_results.csv'
)

mapie_results.to_csv(
    output_file,
    index=False
)

print(
    "Saved:",
    output_file
)

Saved: /content/drive/MyDrive/dissertation_project/data/processed/mapie_conformal_results.csv


In [23]:
text_feature_file = (
    f'{processed_path}/text_features_development.csv'
)

image_feature_file = (
    f'{processed_path}/image_features_development.csv'
)

structured_feature_file = (
    f'{processed_path}/structured_features_mlp.csv'
)

fusion_feature_file = (
    f'{processed_path}/fusion_features_layer1.csv'
)


# LOAD FEATURE CSV FILES

text_df = pd.read_csv(
    text_feature_file
)

image_df = pd.read_csv(
    image_feature_file
)

structured_df = pd.read_csv(
    structured_feature_file
)

fusion_df = pd.read_csv(
    fusion_feature_file
)

print(
    "Text:",
    text_df.shape
)

print(
    "Image:",
    image_df.shape
)

print(
    "Structured:",
    structured_df.shape
)

print(
    "Fusion:",
    fusion_df.shape
)

Text: (1513, 769)
Image: (1513, 769)
Structured: (1513, 129)
Fusion: (1513, 129)


In [24]:
print("\nTEXT COLUMNS:")
print(
    text_df.columns.tolist()[:20]
)

print("\nIMAGE COLUMNS:")
print(
    image_df.columns.tolist()[:20]
)

print("\nSTRUCTURED COLUMNS:")
print(
    structured_df.columns.tolist()[:20]
)

print("\nFUSION COLUMNS:")
print(
    fusion_df.columns.tolist()[:20]
)


TEXT COLUMNS:
['study_id', 'text_feature_0', 'text_feature_1', 'text_feature_2', 'text_feature_3', 'text_feature_4', 'text_feature_5', 'text_feature_6', 'text_feature_7', 'text_feature_8', 'text_feature_9', 'text_feature_10', 'text_feature_11', 'text_feature_12', 'text_feature_13', 'text_feature_14', 'text_feature_15', 'text_feature_16', 'text_feature_17', 'text_feature_18']

IMAGE COLUMNS:
['study_id', 'image_feature_0', 'image_feature_1', 'image_feature_2', 'image_feature_3', 'image_feature_4', 'image_feature_5', 'image_feature_6', 'image_feature_7', 'image_feature_8', 'image_feature_9', 'image_feature_10', 'image_feature_11', 'image_feature_12', 'image_feature_13', 'image_feature_14', 'image_feature_15', 'image_feature_16', 'image_feature_17', 'image_feature_18']

STRUCTURED COLUMNS:
['study_id', 'structured_feature_0', 'structured_feature_1', 'structured_feature_2', 'structured_feature_3', 'structured_feature_4', 'structured_feature_5', 'structured_feature_6', 'structured_feature_

In [25]:
id_columns = [
    'subject_id',
    'study_id',
    'hadm_id',
    'itemid',
    'charttime',
    'label',
    'target',
    'y'
]


def extract_numeric_features(df):

    # Keep only numeric columns
    numeric_df = df.select_dtypes(
        include=[np.number]
    ).copy()

    # Remove identifiers / labels if present
    remove_columns = [
        col
        for col in id_columns
        if col in numeric_df.columns
    ]

    numeric_df = numeric_df.drop(
        columns=remove_columns,
        errors='ignore'
    )

    return numeric_df


X_text = extract_numeric_features(
    text_df
).values.astype(
    np.float32
)

X_image = extract_numeric_features(
    image_df
).values.astype(
    np.float32
)

X_structured = extract_numeric_features(
    structured_df
).values.astype(
    np.float32
)

X_fusion = extract_numeric_features(
    fusion_df
).values.astype(
    np.float32
)


print(
    "X_text:",
    X_text.shape
)

print(
    "X_image:",
    X_image.shape
)

print(
    "X_structured:",
    X_structured.shape
)

print(
    "X_fusion:",
    X_fusion.shape
)

X_text: (1513, 768)
X_image: (1513, 768)
X_structured: (1513, 128)
X_fusion: (1513, 128)


In [26]:
# Use the multimodal fusion representation
X_main = X_fusion.copy()

# Keep the original calibration labels
y_all = np.asarray(
    y_calibration
).copy()

# Align feature and label lengths
n_main = min(
    len(X_main),
    len(y_all),
    len(calibration_df)
)

X_main = X_main[:n_main]
y_all = y_all[:n_main]

# Keep the corresponding metadata rows
calibration_work = (
    calibration_df
    .iloc[:n_main]
    .copy()
    .reset_index(drop=True)
)

# Remove uncertain labels
valid_mask = (
    y_all != -1
)

X_main = X_main[
    valid_mask
]

y_main = y_all[
    valid_mask
]

calibration_work = (
    calibration_work[
        valid_mask
    ]
    .reset_index(drop=True)
)

print(
    "Final conformal records:",
    len(y_main)
)

print(
    "Feature shape:",
    X_main.shape
)

print(
    "\nClass distribution:"
)

print(
    pd.Series(
        y_main
    ).value_counts()
)

Final conformal records: 651
Feature shape: (651, 128)

Class distribution:
0    455
1    196
Name: count, dtype: int64


In [27]:
def prediction_set_metrics(
    y_true,
    prediction_sets
):

    # MAPIE returns a scalar in the installed version
    coverage = float(
        classification_coverage_score(
            y_true,
            prediction_sets
        )
    )

    set_size = float(
        classification_mean_width_score(
            prediction_sets
        )
    )

    efficiency = (
        coverage / set_size
        if set_size > 0
        else np.nan
    )

    return {
        'Coverage': coverage,
        'Average Set Size': set_size,
        'Efficiency': efficiency
    }


def calculate_ece(
    y_true,
    probabilities,
    n_bins=10
):

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )

    confidence = np.max(
        probabilities,
        axis=1
    )

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    # Make sure class labels are 0/1
    predicted_labels = (
        probabilities.argmax(
            axis=1
        )
    )

    classes = np.unique(
        y_true
    )

    if len(classes) == 2:

        predicted_class_labels = np.array([
            classes[p]
            for p in predicted_labels
        ])

        correctness = (
            predicted_class_labels
            == y_true
        ).astype(float)

    else:

        correctness = (
            predictions
            == y_true
        ).astype(float)

    bins = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(
        n_bins
    ):

        lower = bins[i]
        upper = bins[i + 1]

        if i == n_bins - 1:

            mask = (
                (confidence >= lower)
                &
                (confidence <= upper)
            )

        else:

            mask = (
                (confidence >= lower)
                &
                (confidence < upper)
            )

        if mask.sum() == 0:

            continue

        bin_accuracy = np.mean(
            correctness[mask]
        )

        bin_confidence = np.mean(
            confidence[mask]
        )

        bin_weight = (
            mask.sum()
            / len(y_true)
        )

        ece += (
            bin_weight
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)

In [28]:
# NON-CONFORMITY SCORES

base_model = LogisticRegression(
    max_iter=500,
    random_state=42
)

base_model.fit(
    X_main,
    y_main
)

main_probabilities = (
    base_model.predict_proba(
        X_main
    )
)

classes = (
    base_model.classes_
)

class_to_index = {
    cls: idx
    for idx, cls in enumerate(
        classes
    )
}

true_class_probability = np.array([

    main_probabilities[
        i,
        class_to_index[
            y_main[i]
        ]
    ]

    for i in range(
        len(y_main)
    )

])

nonconformity_scores = (
    1
    - true_class_probability
)

print(
    "Mean non-conformity:",
    round(
        np.mean(
            nonconformity_scores
        ),
        6
    )
)

print(
    "Non-conformity spread:",
    round(
        np.std(
            nonconformity_scores
        ),
        6
    )
)

Mean non-conformity: 0.35952
Non-conformity spread: 0.223887


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [29]:
# SPLIT CONFORMAL

confidence_level = 0.90

split_estimator = LogisticRegression(
    max_iter=500,
    random_state=42
)

split_estimator.fit(
    X_main,
    y_main
)

split_mapie = SplitConformalClassifier(
    estimator=split_estimator,
    confidence_level=confidence_level,
    conformity_score="lac",
    prefit=True
)

split_mapie.conformalize(
    X_main,
    y_main
)

split_pred, split_sets = (
    split_mapie.predict_set(
        X_main
    )
)

split_metrics = prediction_set_metrics(
    y_main,
    split_sets
)

split_probabilities = (
    split_estimator.predict_proba(
        X_main
    )
)

split_ece = calculate_ece(
    y_main,
    split_probabilities
)

split_mean_nc = float(
    np.mean(
        split_mapie.conformity_scores
    )
)

split_std_nc = float(
    np.std(
        split_mapie.conformity_scores
    )
)

print(
    "\nSPLIT CONFORMAL"
)

print(
    "Coverage:",
    round(
        split_metrics['Coverage'],
        6
    )
)

print(
    "Average set size:",
    round(
        split_metrics[
            'Average Set Size'
        ],
        6
    )
)

print(
    "Efficiency:",
    round(
        split_metrics[
            'Efficiency'
        ],
        6
    )
)

print(
    "Mean non-conformity:",
    round(
        split_mean_nc,
        6
    )
)

print(
    "Non-conformity spread:",
    round(
        split_std_nc,
        6
    )
)

print(
    "ECE:",
    round(
        split_ece,
        6
    )
)


SPLIT CONFORMAL
Coverage: 0.90169
Average set size: 1.425499
Efficiency: 0.632543
Mean non-conformity: 0.35952
Non-conformity spread: 0.223887
ECE: 0.024871


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/tmp/ipykernel_1045/3621258480.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(
/tmp/ipykernel_1045/3621258480.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before perf

In [30]:
# CV+

cv_estimator = LogisticRegression(
    max_iter=500,
    random_state=42
)

cv_mapie = CrossConformalClassifier(
    estimator=cv_estimator,
    confidence_level=confidence_level,
    conformity_score="lac",
    cv=5,
    n_jobs=-1,
    random_state=42
)

cv_mapie.fit_conformalize(
    X_main,
    y_main
)

cv_pred, cv_sets = (
    cv_mapie.predict_set(
        X_main,
        agg_scores="mean"
    )
)

cv_metrics = prediction_set_metrics(
    y_main,
    cv_sets
)

cv_base_model = LogisticRegression(
    max_iter=500,
    random_state=42
)

cv_base_model.fit(
    X_main,
    y_main
)

cv_probabilities = (
    cv_base_model.predict_proba(
        X_main
    )
)

cv_ece = calculate_ece(
    y_main,
    cv_probabilities
)

cv_mean_nc = float(
    np.mean(
        cv_mapie.conformity_scores
    )
)

cv_std_nc = float(
    np.std(
        cv_mapie.conformity_scores
    )
)

print(
    "\nCV+ / CROSS-CONFORMAL"
)

print(
    "Coverage:",
    round(
        cv_metrics['Coverage'],
        6
    )
)

print(
    "Average set size:",
    round(
        cv_metrics[
            'Average Set Size'
        ],
        6
    )
)

print(
    "Efficiency:",
    round(
        cv_metrics[
            'Efficiency'
        ],
        6
    )
)

print(
    "Mean non-conformity:",
    round(
        cv_mean_nc,
        6
    )
)

print(
    "Non-conformity spread:",
    round(
        cv_std_nc,
        6
    )
)

print(
    "ECE:",
    round(
        cv_ece,
        6
    )
)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



CV+ / CROSS-CONFORMAL
Coverage: 0.990783
Average set size: 1.864823
Efficiency: 0.531301
Mean non-conformity: 0.435666
Non-conformity spread: 0.294766
ECE: 0.024871


/tmp/ipykernel_1045/3621258480.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(
/tmp/ipykernel_1045/3621258480.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  set_size = float(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_

In [31]:
# AGE INFORMATION

age_candidates = [
    'anchor_age',
    'age',
    'patient_age',
    'age_years'
]

age_column = None

for candidate in age_candidates:

    if candidate in calibration_work.columns:

        age_column = candidate
        break


print(
    "Age column:",
    age_column
)

if age_column is not None:

    age_values = pd.to_numeric(
        calibration_work[
            age_column
        ],
        errors='coerce'
    )

    print(
        "\nAge summary:"
    )

    print(
        age_values.describe()
    )

    print(
        "\nUnique age values:",
        age_values.nunique()
    )

else:

    age_values = None

    print(
        "No age information found."
    )

Age column: anchor_age

Age summary:
count    651.000000
mean       0.042445
std        0.985832
min       -2.612309
25%       -0.603500
50%        0.066103
75%        0.796578
max        1.709673
Name: anchor_age, dtype: float64

Unique age values: 70


In [32]:
# CREATE TWO AGE BANDS

if age_values is not None:

    valid_age = (
        age_values
        .notna()
    )

    median_age = (
        age_values[
            valid_age
        ]
        .median()
    )

    age_band = pd.Series(
        index=age_values.index,
        dtype='object'
    )

    age_band[
        age_values <= median_age
    ] = (
        f'Age band 1: <= {int(median_age)}'
    )

    age_band[
        age_values > median_age
    ] = (
        f'Age band 2: > {int(median_age)}'
    )

    print(
        "Median age:",
        median_age
    )

    print(
        "\nAge band distribution:"
    )

    print(
        age_band.value_counts(
            dropna=False
        )
    )

else:

    age_band = None

Median age: 0.0661025296662653

Age band distribution:
Age band 1: <= 0    331
Age band 2: > 0     320
Name: count, dtype: int64


In [33]:
# MONDRIAN

def mondrian_prediction_sets(
    X,
    y,
    groups,
    confidence_level=0.90
):

    model = LogisticRegression(
        max_iter=500,
        random_state=42
    )

    model.fit(
        X,
        y
    )

    probabilities = (
        model.predict_proba(
            X
        )
    )

    classes = model.classes_

    class_to_idx = {
        cls: idx
        for idx, cls in enumerate(
            classes
        )
    }

    prediction_sets = np.zeros(
        (
            len(X),
            len(classes)
        ),
        dtype=bool
    )

    group_quantiles = {}

    groups_array = np.asarray(
        groups
    )

    for group in pd.Series(
        groups_array
    ).dropna().unique():

        group_mask = (
            groups_array
            == group
        )

        group_indices = np.where(
            group_mask
        )[0]

        if len(group_indices) < 5:

            continue

        group_probabilities = (
            probabilities[
                group_indices
            ]
        )

        group_y = (
            y[
                group_indices
            ]
        )

        true_probabilities = np.array([

            group_probabilities[
                i,
                class_to_idx[
                    group_y[i]
                ]
            ]

            for i in range(
                len(group_y)
            )

        ])

        scores = (
            1
            - true_probabilities
        )

        # Finite-sample conformal quantile
        alpha = (
            1
            - confidence_level
        )

        q_level = (
            np.ceil(
                (len(scores) + 1)
                * (1 - alpha)
            )
            / len(scores)
        )

        q_level = min(
            q_level,
            1.0
        )

        q = np.quantile(
            scores,
            q_level,
            method='higher'
        )

        group_quantiles[
            str(group)
        ] = float(q)

        # Generate prediction sets
        for row_index in group_indices:

            row_scores = (
                1
                - probabilities[
                    row_index
                ]
            )

            prediction_sets[
                row_index
            ] = (
                row_scores <= q
            )

    return (
        model,
        prediction_sets,
        group_quantiles
    )


(
    mondrian_model,
    mondrian_sets,
    mondrian_quantiles
) = mondrian_prediction_sets(

    X_main,

    y_main,

    age_band,

    confidence_level
)

mondrian_metrics = (
    prediction_set_metrics(
        y_main,
        mondrian_sets
    )
)

mondrian_probabilities = (
    mondrian_model.predict_proba(
        X_main
    )
)

mondrian_ece = calculate_ece(
    y_main,
    mondrian_probabilities
)

# Non-conformity statistics
mondrian_true_prob = np.array([

    mondrian_probabilities[
        i,
        class_to_index[
            y_main[i]
        ]
    ]

    for i in range(
        len(y_main)
    )

])

mondrian_nc = (
    1
    - mondrian_true_prob
)

mondrian_mean_nc = float(
    np.mean(
        mondrian_nc
    )
)

mondrian_std_nc = float(
    np.std(
        mondrian_nc
    )
)

print(
    "\nMONDRIAN RESULTS"
)

print(
    "Coverage:",
    round(
        mondrian_metrics[
            'Coverage'
        ],
        6
    )
)

print(
    "Average set size:",
    round(
        mondrian_metrics[
            'Average Set Size'
        ],
        6
    )
)

print(
    "Efficiency:",
    round(
        mondrian_metrics[
            'Efficiency'
        ],
        6
    )
)

print(
    "Mean non-conformity:",
    round(
        mondrian_mean_nc,
        6
    )
)

print(
    "Non-conformity spread:",
    round(
        mondrian_std_nc,
        6
    )
)

print(
    "ECE:",
    round(
        mondrian_ece,
        6
    )
)

print(
    "\nMondrian group quantiles:"
)

print(
    mondrian_quantiles
)


MONDRIAN RESULTS
Coverage: 0.906298
Average set size: 1.451613
Efficiency: 0.624339
Mean non-conformity: 0.35952
Non-conformity spread: 0.223887
ECE: 0.024871

Mondrian group quantiles:
{'Age band 2: > 0': 0.7240861636099809, 'Age band 1: <= 0': 0.6982743593357891}


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/tmp/ipykernel_1045/3621258480.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(


In [34]:
# SUBGROUP
def calculate_group_coverage(
    y_true,
    prediction_sets,
    groups
):

    output = []

    groups_series = pd.Series(
        groups
    )

    for group in (
        groups_series
        .dropna()
        .unique()
    ):

        mask = (
            groups_series
            == group
        ).to_numpy()

        if mask.sum() == 0:

            continue

        coverage = float(
            classification_coverage_score(
                y_true[mask],
                prediction_sets[mask]
            )
        )

        output.append({

            'Subgroup': str(group),

            'N': int(
                mask.sum()
            ),

            'Coverage': coverage
        })

    return pd.DataFrame(
        output
    )

In [35]:
# AGE-BAND

split_age_coverage = (
    calculate_group_coverage(
        y_main,
        split_sets,
        age_band
    )
)

cv_age_coverage = (
    calculate_group_coverage(
        y_main,
        cv_sets,
        age_band
    )
)

mondrian_age_coverage = (
    calculate_group_coverage(
        y_main,
        mondrian_sets,
        age_band
    )
)

print(
    "\nSPLIT AGE COVERAGE"
)

display(
    split_age_coverage
)

print(
    "\nCV+ AGE COVERAGE"
)

display(
    cv_age_coverage
)

print(
    "\nMONDRIAN AGE COVERAGE"
)

display(
    mondrian_age_coverage
)


SPLIT AGE COVERAGE


/tmp/ipykernel_1045/3977690983.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(
/tmp/ipykernel_1045/3977690983.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(
/tmp/ipykernel_1045/3977690983.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(
/tmp/ipykernel_1045/3977690983.py:29: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extr

,Subgroup,N,Coverage
0,Age band 2: > 0,320,0.893750
1,Age band 1: <= 0,331,0.909366



CV+ AGE COVERAGE


,Subgroup,N,Coverage
0,Age band 2: > 0,320,0.990625
1,Age band 1: <= 0,331,0.990937



MONDRIAN AGE COVERAGE


,Subgroup,N,Coverage
0,Age band 2: > 0,320,0.906250
1,Age band 1: <= 0,331,0.906344


In [36]:
# RESULTS TABLE

final_conformal_results = pd.DataFrame({

    'Metric': [

        'Non-conformity score (mean)',

        'Non-conformity score (spread/std)',

        'Empirical Coverage',

        'Average Prediction Set Size',

        'Prediction Set Efficiency',

        'Expected Calibration Error (ECE)'
    ],

    'Split': [

        split_mean_nc,

        split_std_nc,

        split_metrics[
            'Coverage'
        ],

        split_metrics[
            'Average Set Size'
        ],

        split_metrics[
            'Efficiency'
        ],

        split_ece
    ],

    'CV+': [

        cv_mean_nc,

        cv_std_nc,

        cv_metrics[
            'Coverage'
        ],

        cv_metrics[
            'Average Set Size'
        ],

        cv_metrics[
            'Efficiency'
        ],

        cv_ece
    ],

    'Mondrian': [

        mondrian_mean_nc,

        mondrian_std_nc,

        mondrian_metrics[
            'Coverage'
        ],

        mondrian_metrics[
            'Average Set Size'
        ],

        mondrian_metrics[
            'Efficiency'
        ],

        mondrian_ece
    ]
})

display(
    final_conformal_results
)

,Metric,Split,CV+,Mondrian
0,Non-conformity score (mean),0.359520,0.435666,0.359520
1,Non-conformity score (spread/std),0.223887,0.294766,0.223887
2,Empirical Coverage,0.901690,0.990783,0.906298
3,Average Prediction Set Size,1.425499,1.864823,1.451613
4,Prediction Set Efficiency,0.632543,0.531301,0.624339
5,Expected Calibration Error (ECE),0.024871,0.024871,0.024871


In [37]:
# ============================================
# VERSION 1: Original (row-position truncation)
# ============================================
fusion_features_dev = pd.read_csv(f'{processed_path}/fusion_features_layer1.csv')
X_fusion_old = fusion_features_dev.select_dtypes(include=[np.number]).copy()

y_calibration_old = (
    calibration_df['Pleural Effusion']
    .fillna(0)
    .astype(int)
    .values
)

n_main_old = min(len(X_fusion_old), len(y_calibration_old), len(calibration_df))
X_main_old = X_fusion_old.iloc[:n_main_old].values
y_main_old = y_calibration_old[:n_main_old]

# remove uncertain (-1) labels, same as your original code
valid_mask_old = (y_main_old != -1)
X_main_old = X_main_old[valid_mask_old]
y_main_old = y_main_old[valid_mask_old]

split_estimator_old = LogisticRegression(max_iter=500, random_state=42)
split_estimator_old.fit(X_main_old, y_main_old)

split_mapie_old = SplitConformalClassifier(
    estimator=split_estimator_old, confidence_level=0.90,
    conformity_score="lac", prefit=True
)
split_mapie_old.conformalize(X_main_old, y_main_old)
_, split_sets_old = split_mapie_old.predict_set(X_main_old)

metrics_old = prediction_set_metrics(y_main_old, split_sets_old)
probs_old = split_estimator_old.predict_proba(X_main_old)
ece_old = calculate_ece(y_main_old, probs_old)

# ============================================
# VERSION 2: Corrected (merged by study_id)
# ============================================
fusion_heldout_df = pd.read_csv(f'{processed_path}/fusion_features_layer1_heldout.csv')
fusion_heldout_df['study_id'] = fusion_heldout_df['study_id'].astype(str)
calibration_df['study_id'] = calibration_df['study_id'].astype(str)

merged = calibration_df.merge(fusion_heldout_df, on='study_id', how='inner')
fusion_cols = [c for c in fusion_heldout_df.columns if c != 'study_id']

y_new = merged['Pleural Effusion'].fillna(0).astype(int).values
valid_mask_new = (y_new != -1)

X_main_new = merged.loc[valid_mask_new, fusion_cols].values.astype(np.float32)
y_main_new = y_new[valid_mask_new]

split_estimator_new = LogisticRegression(max_iter=500, random_state=42)
split_estimator_new.fit(X_main_new, y_main_new)

split_mapie_new = SplitConformalClassifier(
    estimator=split_estimator_new, confidence_level=0.90,
    conformity_score="lac", prefit=True
)
split_mapie_new.conformalize(X_main_new, y_main_new)
_, split_sets_new = split_mapie_new.predict_set(X_main_new)

metrics_new = prediction_set_metrics(y_main_new, split_sets_new)
probs_new = split_estimator_new.predict_proba(X_main_new)
ece_new = calculate_ece(y_main_new, probs_new)

# ============================================
# COMPARISON TABLE
# ============================================
comparison = pd.DataFrame({
    'Metric': ['N records', 'Coverage', 'Average Set Size', 'Efficiency', 'ECE'],
    'Original (row truncation)': [
        len(y_main_old), metrics_old['Coverage'], metrics_old['Average Set Size'],
        metrics_old['Efficiency'], ece_old
    ],
    'Corrected (study_id merge)': [
        len(y_main_new), metrics_new['Coverage'], metrics_new['Average Set Size'],
        metrics_new['Efficiency'], ece_new
    ]
})
display(comparison)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/tmp/ipykernel_1045/3621258480.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  coverage = float(
/tmp/ipykernel_1045/3621258480.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before perf

,Metric,Original (row truncation),Corrected (study_id merge)
0,N records,651.000000,651.000000
1,Coverage,0.901690,0.901690
2,Average Set Size,1.505376,0.964670
3,Efficiency,0.598980,0.934713
4,ECE,0.012483,0.016242
